# Notebook 06: Information-Theoretic Analysis (Level 5)

## Overview
This notebook quantifies **how much information about frequency band identity**
is encoded in Pythia model activations at each layer, using information-theoretic
measures. While previous notebooks used geometric (NB01, NB02) and probing
(NB02) approaches, this notebook provides a complementary perspective grounded
in mutual information (MI), conditional entropy, and coding efficiency.

## Key Questions
- How many bits of band information are encoded in residual stream activations at each layer?
- At which layer does band information peak, and how does this relate to probe accuracy peaks?
- Which layers contribute the most new band information (delta-MI)?
- Do attention and MLP components carry different amounts of band information?
- How efficiently is band information encoded (bits per dimension)?
- How do MI trajectories scale with model size?
- Do information-theoretic measures correlate with geometric measures from NB01/NB02?

## Hypothesis Domain: R5 (Information-Theoretic)
- **H-R5.1**: MI(activations; band) increases through layers (Jonckheere-Terpstra, per model)
- **H-R5.2**: Delta-MI peaks in early-to-mid layers (peak location test, per model)
- **H-R5.3**: MLP components carry more band MI than attention at information-critical layers
- **H-R5.4**: Coding efficiency (bits/dim) increases with model size (Spearman across models)
- **H-R5.5**: KSG MI and probe-based MI are strongly correlated (Pearson, pooled)
- **H-R5.6**: MI trajectory correlates with probe accuracy trajectory (Pearson, per model)

## Notebook Structure
1. Setup & Data Loading
2. MI(activations; band) per layer (KSG + probe-based)
3. Conditional entropy H(band | activations) per layer
4. Delta-MI per layer (information gain)
5. Component-wise MI (attention vs MLP)
6. Coding efficiency (bits/dim)
7. Cross-model comparison
8. Integration with geometric analyses (NB01/NB02)
9. Summary

## Data Sources
- Pre-extracted activations: `resid_post_predpos` (N, n_layers, d_model)
- Pre-extracted activations: `attn_out_predpos` (N, n_layers, d_model)
- Pre-extracted activations: `mlp_out_predpos` (N, n_layers, d_model)
- NB02 results: `02_probe_trajectory.csv`, `02_separation_trajectory.csv`

## 1. Setup & Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    ACTIVATIONS_DIR,
    ANALYSIS_DIR,
    VIZ_DIR,
    get_domain_dirs,
    K_NEIGHBORS,
    RANDOM_SEED,
)
from utils.data_loading import load_extracted_activations, save_analysis
from utils.info_theory import (
    estimate_mi_ksg,
    probe_based_mi,
    mi_from_accuracy,
    compute_conditional_entropy,
    compute_mi_trajectory,
    compute_delta_mi,
    compute_coding_efficiency_trajectory,
    compute_component_mi,
)
from utils.probing import train_probe
from utils.plotting import (
    setup_plotting,
    save_figure,
    plot_mi_trajectory,
    plot_metric_heatmap,
)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy import stats as sp_stats

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_domain_dirs("info_theoretic", "base")
from functools import partial as _partial

save_analysis = _partial(save_analysis, analysis_dir=ANALYSIS_DIR)
save_figure = _partial(save_figure, viz_dir=VIZ_DIR)

print(f"Models: {MODELS}")
print(f"Bands:  {BANDS}")
print(f"Draws:  {DRAWS}")
print(f"K_NEIGHBORS: {K_NEIGHBORS}")
print(f"Analysis dir: {ANALYSIS_DIR}")
print(f"Viz dir: {VIZ_DIR}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands:  ['low', 'medium', 'high', 'very_high', 'control']
Draws:  ['draw_1', 'draw_2', 'draw_3']
K_NEIGHBORS: 10
Analysis dir: LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/base/analysis
Viz dir: LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/base/viz


In [2]:
# Load residual stream, attention output, and MLP output activations
# resid_post_predpos: (N, n_layers, d_model)
# attn_out_predpos:   (N, n_layers, d_model)
# mlp_out_predpos:    (N, n_layers, d_model)

all_resid = {}  # model -> draw -> {band: (N, n_layers, d_model)}
all_attn_out = {}  # model -> draw -> {band: (N, n_layers, d_model)}
all_mlp_out = {}  # model -> draw -> {band: (N, n_layers, d_model)}

for model in MODELS:
    all_resid[model] = {}
    all_attn_out[model] = {}
    all_mlp_out[model] = {}
    for draw in DRAWS:
        all_resid[model][draw] = {}
        all_attn_out[model][draw] = {}
        all_mlp_out[model][draw] = {}
        for band in BANDS:
            try:
                data = load_extracted_activations(model, band, draw)
                all_resid[model][draw][band] = data["resid_post_predpos"]
                if "attn_out_predpos" in data:
                    all_attn_out[model][draw][band] = data["attn_out_predpos"]
                if "mlp_out_predpos" in data:
                    all_mlp_out[model][draw][band] = data["mlp_out_predpos"]
            except FileNotFoundError:
                pass

# Report shapes
for model in MODELS:
    sample = next(iter(next(iter(all_resid[model].values())).values()), None)
    if sample is not None:
        print(f"{model}: resid shape = {sample.shape}  (N, n_layers, d_model)")
    attn_sample = next(iter(next(iter(all_attn_out[model].values())).values()), None)
    if attn_sample is not None:
        print(f"  attn_out shape = {attn_sample.shape}")
    mlp_sample = next(iter(next(iter(all_mlp_out[model].values())).values()), None)
    if mlp_sample is not None:
        print(f"  mlp_out shape  = {mlp_sample.shape}")

pythia-70m: resid shape = (225, 6, 512)  (N, n_layers, d_model)
  attn_out shape = (225, 6, 512)
  mlp_out shape  = (225, 6, 512)
pythia-160m: resid shape = (225, 12, 768)  (N, n_layers, d_model)
  attn_out shape = (225, 12, 768)
  mlp_out shape  = (225, 12, 768)
pythia-410m: resid shape = (225, 24, 1024)  (N, n_layers, d_model)
  attn_out shape = (225, 24, 1024)
  mlp_out shape  = (225, 24, 1024)
pythia-1b: resid shape = (225, 16, 2048)  (N, n_layers, d_model)
  attn_out shape = (225, 16, 2048)
  mlp_out shape  = (225, 16, 2048)
pythia-1.4b: resid shape = (225, 24, 2048)  (N, n_layers, d_model)
  attn_out shape = (225, 24, 2048)
  mlp_out shape  = (225, 24, 2048)


In [3]:
# Helper: build combined activations and labels for a given model, draw, layer


def build_layer_data(activation_dict, model, draw, layer):
    """Combine bands into a single (X, labels) pair for a specific layer.

    Args:
        activation_dict: e.g. all_resid[model][draw]
        model: Model name.
        draw: Draw name.
        layer: Layer index.

    Returns:
        (X, labels) where X is (N_total, d_model) and labels is (N_total,),
        or (None, None) if insufficient data.
    """
    embs, labels = [], []
    for band in BANDS:
        act = activation_dict.get(model, {}).get(draw, {}).get(band)
        if act is not None:
            embs.append(act[:, layer, :])
            labels.extend([band] * act.shape[0])
    if len(embs) < 2:
        return None, None
    return np.vstack(embs), np.array(labels)


def reduce_dims(X, n_components=10):
    """Apply PCA dimensionality reduction if needed."""
    if X.shape[1] > n_components:
        pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
        return pca.fit_transform(X)
    return X


n_classes = len(BANDS)
H_Y = np.log2(n_classes)  # Max entropy assuming uniform prior
print(f"Number of classes: {n_classes}")
print(f"H(Y) = log2({n_classes}) = {H_Y:.3f} bits (uniform prior upper bound)")

Number of classes: 5
H(Y) = log2(5) = 2.322 bits (uniform prior upper bound)


## 2. MI(activations; band) per Layer

Two complementary estimators for robustness:
1. **KSG estimator** (Kraskov et al., 2004): non-parametric, based on k-NN distances
2. **Probe-based MI**: MI lower bound derived from linear probe predictions

PCA to 10 dims before KSG estimation to manage computational cost for high-dimensional models.

In [4]:
# KSG MI estimation at each layer
mi_ksg_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_model = MODEL_D_MODEL[model]
    print(f"\n{model} ({n_layers} layers, d_model={d_model}):")

    for draw in DRAWS:
        for layer in range(n_layers):
            X, labels = build_layer_data(all_resid, model, draw, layer)
            if X is None:
                continue

            # Reduce dimensionality for KSG if needed
            X_reduced = reduce_dims(X, n_components=10)

            mi = estimate_mi_ksg(X_reduced, labels, k=K_NEIGHBORS)

            mi_ksg_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "mi_ksg": mi,
                    "method": "ksg",
                    "d_model": d_model,
                }
            )

        # Progress report (draw_1 only)
        if draw == "draw_1":
            draw_records = [
                r for r in mi_ksg_records if r["model"] == model and r["draw"] == draw
            ]
            if draw_records:
                peak = max(draw_records, key=lambda r: r["mi_ksg"])
                print(
                    f"  {draw}: peak MI(KSG) = {peak['mi_ksg']:.3f} bits at layer {peak['layer']}"
                )

df_mi_ksg = pd.DataFrame(mi_ksg_records)
save_analysis(df_mi_ksg, "06_mi_ksg_trajectory.csv")
print(f"\nKSG MI records: {len(df_mi_ksg)}")


pythia-70m (6 layers, d_model=512):


  draw_1: peak MI(KSG) = 0.759 bits at layer 0



pythia-160m (12 layers, d_model=768):


  draw_1: peak MI(KSG) = 0.907 bits at layer 6



pythia-410m (24 layers, d_model=1024):


...


pythia-1b (16 layers, d_model=2048):


  draw_1: peak MI(KSG) = 1.152 bits at layer 14



pythia-1.4b (24 layers, d_model=2048):


  draw_1: peak MI(KSG) = 1.123 bits at layer 22



KSG MI records: 246


In [5]:
# Probe-based MI estimation at ALL layers for ALL draws
# Probe-MI is the PRIMARY information metric because KSG MI is unreliable
# in high dimensions (d=512-2048 with N=1125). KSG is retained as supplementary.
mi_probe_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model} ({n_layers} layers):")

    for draw in DRAWS:
        for layer in range(n_layers):
            X, labels = build_layer_data(all_resid, model, draw, layer)
            if X is None:
                continue

            # Train linear probe and get predictions
            probe_result = train_probe(X, labels, return_predictions=True)
            predictions = probe_result["predictions"]
            true_labels = probe_result["true_labels"]

            # Probe-based MI from confusion matrix
            mi_probe = probe_based_mi(predictions, true_labels)

            # Fano-bound MI from accuracy
            mi_fano = mi_from_accuracy(probe_result["accuracy"], n_classes)

            mi_probe_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "mi_probe": mi_probe,
                    "mi_fano": mi_fano,
                    "probe_accuracy": probe_result["accuracy"],
                    "method": "probe",
                }
            )

        # Progress report
        draw_records = [
            r for r in mi_probe_records if r["model"] == model and r["draw"] == draw
        ]
        if draw_records:
            peak = max(draw_records, key=lambda r: r["mi_probe"])
            print(
                f"  {draw}: peak MI(probe) = {peak['mi_probe']:.3f} bits at layer {peak['layer']}"
            )

df_mi_probe = pd.DataFrame(mi_probe_records)
save_analysis(df_mi_probe, "06_mi_probe_trajectory.csv")
print(f"\nProbe-based MI records: {len(df_mi_probe)}")
print(
    f"\nNOTE: Probe-MI is the primary metric. KSG MI (cell above) is supplementary only."
)
print(f"KSG MI is known to fail in high dimensions (d >> N). With d_model=512-2048 and")
print(
    f"N=1125, even PCA to 10 dims produces noisy estimates. Probe-MI gives monotonically"
)
print(f"increasing trajectories consistent with probe accuracy, while KSG MI peaks at")
print(f"layer 0 for some models (contradicting probe accuracy increase).")


pythia-70m (6 layers):


  draw_1: peak MI(probe) = 0.967 bits at layer 3


  draw_2: peak MI(probe) = 0.935 bits at layer 2


  draw_3: peak MI(probe) = 0.908 bits at layer 4

pythia-160m (12 layers):


  draw_1: peak MI(probe) = 1.147 bits at layer 7


...

  draw_2: peak MI(probe) = 1.281 bits at layer 13


  draw_3: peak MI(probe) = 1.211 bits at layer 13

pythia-1.4b (24 layers):


  draw_1: peak MI(probe) = 1.353 bits at layer 20


  draw_2: peak MI(probe) = 1.395 bits at layer 21


  draw_3: peak MI(probe) = 1.367 bits at layer 22

Probe-based MI records: 246

NOTE: Probe-MI is the primary metric. KSG MI (cell above) is supplementary only.
KSG MI is known to fail in high dimensions (d >> N). With d_model=512-2048 and
N=1125, even PCA to 10 dims produces noisy estimates. Probe-MI gives monotonically
increasing trajectories consistent with probe accuracy, while KSG MI peaks at
layer 0 for some models (contradicting probe accuracy increase).


In [6]:
# Merge KSG and probe MI into a combined DataFrame
# Probe MI is now available at ALL layers for ALL draws
df_mi_combined = df_mi_ksg[["model", "draw", "layer", "mi_ksg"]].merge(
    df_mi_probe[["model", "draw", "layer", "mi_probe", "mi_fano", "probe_accuracy"]],
    on=["model", "draw", "layer"],
    how="outer",
)
save_analysis(df_mi_combined, "06_mi_combined_trajectory.csv")
print(f"Combined MI records: {len(df_mi_combined)}")
print(df_mi_combined.head(10))

# Report KSG vs probe MI correlation
valid = df_mi_combined.dropna(subset=["mi_ksg", "mi_probe"])
if len(valid) > 5:
    r, p = sp_stats.pearsonr(valid["mi_ksg"], valid["mi_probe"])
    print(f"\nKSG vs Probe MI correlation: r={r:.3f}, p={p:.2e} (n={len(valid)})")
    if r < 0.5:
        print(
            "  WARNING: Low correlation confirms KSG MI is unreliable in these dimensions."
        )

Combined MI records: 246
         model    draw  layer    mi_ksg  mi_probe   mi_fano  probe_accuracy
0  pythia-1.4b  draw_1      0  0.995680  0.867282  0.490659        0.576000
1  pythia-1.4b  draw_1      1  1.102659  0.947635  0.546392        0.598222
2  pythia-1.4b  draw_1      2  1.067508  0.978408  0.571850        0.608000
3  pythia-1.4b  draw_1      3  1.047809  0.969882  0.562526        0.604444
4  pythia-1.4b  draw_1      4  1.013993  0.970180  0.553278        0.600889
5  pythia-1.4b  draw_1      5  0.914534  0.978806  0.562526        0.604444
6  pythia-1.4b  draw_1      6  0.929235  0.906654  0.523746        0.589333
7  pythia-1.4b  draw_1      7  0.785251  0.939876  0.539548        0.595556
8  pythia-1.4b  draw_1      8  0.824828  0.998194  0.614760        0.624000
9  pythia-1.4b  draw_1      9  0.741684  0.956746  0.597887        0.617778

KSG vs Probe MI correlation: r=0.441, p=4.00e-13 (n=246)


### Visualization: MI Trajectories (KSG vs Probe-based)

In [7]:
# Plot MI trajectories per model: KSG vs probe-based
for model in MODELS:
    model_data = df_mi_combined[
        (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(
        model_data["layer"],
        model_data["mi_ksg"],
        color="#1f77b4",
        label="MI (KSG)",
        marker="o",
        markersize=4,
    )
    ax.plot(
        model_data["layer"],
        model_data["mi_probe"],
        color="#ff7f0e",
        label="MI (Probe-based)",
        marker="s",
        markersize=4,
    )
    ax.plot(
        model_data["layer"],
        model_data["mi_fano"],
        color="#2ca02c",
        label="MI (Fano bound)",
        marker="^",
        markersize=4,
        linestyle="--",
        alpha=0.7,
    )

    ax.axhline(
        y=H_Y, color="gray", linestyle=":", alpha=0.5, label=f"H(Y)={H_Y:.2f} bits"
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mutual Information (bits)")
    ax.set_title(f"MI(activations; band) Trajectory -- {model}")
    ax.legend()
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    save_figure(fig, f"viz_06_01_mi_trajectory_{model}.png")

In [8]:
# All models on one plot (KSG MI, draw_1)
fig, ax = plt.subplots(figsize=(14, 7))
for model in MODELS:
    model_data = df_mi_combined[
        (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    n_layers = MODEL_INFO[model]["n_layers"]
    # Normalize layer to [0, 1] for cross-model comparison
    layer_frac = model_data["layer"] / (n_layers - 1)
    ax.plot(
        layer_frac,
        model_data["mi_ksg"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

ax.axhline(y=H_Y, color="gray", linestyle=":", alpha=0.5, label=f"H(Y)={H_Y:.2f}")
ax.set_xlabel("Relative Layer Position (0=first, 1=last)")
ax.set_ylabel("MI (KSG, bits)")
ax.set_title("MI(activations; band) Across Models (Normalized Depth)")
ax.legend()
ax.set_ylim(bottom=0)
fig.tight_layout()
save_figure(fig, "viz_06_02_mi_trajectory_all_models.png")

In [9]:
# Correlation between KSG and probe-based MI
valid = df_mi_combined.dropna(subset=["mi_ksg", "mi_probe"])
if len(valid) > 5:
    r, p = sp_stats.pearsonr(valid["mi_ksg"], valid["mi_probe"])
    rho, p_rho = sp_stats.spearmanr(valid["mi_ksg"], valid["mi_probe"])
    print(f"KSG vs Probe MI correlation (all models/draws/layers):")
    print(f"  Pearson r = {r:.4f}, p = {p:.2e}")
    print(f"  Spearman rho = {rho:.4f}, p = {p_rho:.2e}")

    fig, ax = plt.subplots(figsize=(8, 8))
    for model in MODELS:
        md = valid[valid["model"] == model]
        ax.scatter(
            md["mi_ksg"],
            md["mi_probe"],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            alpha=0.6,
            s=30,
        )
    lims = [0, max(valid["mi_ksg"].max(), valid["mi_probe"].max()) * 1.1]
    ax.plot(lims, lims, "k--", alpha=0.3, label="y=x")
    ax.set_xlabel("MI (KSG, bits)")
    ax.set_ylabel("MI (Probe-based, bits)")
    ax.set_title(f"KSG vs Probe MI (Pearson r={r:.3f})")
    ax.legend()
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    fig.tight_layout()
    save_figure(fig, "viz_06_03_mi_ksg_vs_probe_scatter.png")
else:
    print("Insufficient data for MI correlation analysis.")

KSG vs Probe MI correlation (all models/draws/layers):
  Pearson r = 0.4409, p = 4.00e-13
  Spearman rho = 0.4101, p = 2.14e-11


## 3. Conditional Entropy H(band | activations) per Layer

H(Y | X) = H(Y) - MI(X; Y). This measures the remaining uncertainty about
band identity after observing the activations. Lower values mean the activations
are more informative about band membership.

In [10]:
cond_ent_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model}:")

    for draw in DRAWS:
        for layer in range(n_layers):
            X, labels = build_layer_data(all_resid, model, draw, layer)
            if X is None:
                continue

            X_reduced = reduce_dims(X, n_components=10)
            h_cond = compute_conditional_entropy(X_reduced, labels, k=K_NEIGHBORS)

            # Also compute marginal H(Y) from label distribution
            unique, counts = np.unique(labels, return_counts=True)
            p_y = counts / len(labels)
            h_y = -np.sum(p_y * np.log2(np.clip(p_y, 1e-10, 1.0)))

            cond_ent_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "h_y": h_y,
                    "h_y_given_x": h_cond,
                    "info_fraction": 1.0 - (h_cond / h_y) if h_y > 0 else 0.0,
                }
            )

        if draw == "draw_1":
            draw_recs = [
                r for r in cond_ent_records if r["model"] == model and r["draw"] == draw
            ]
            if draw_recs:
                best = min(draw_recs, key=lambda r: r["h_y_given_x"])
                print(
                    f"  {draw}: min H(Y|X) = {best['h_y_given_x']:.3f} bits at layer {best['layer']}"
                )
                print(f"          info fraction = {best['info_fraction']:.3f}")

df_cond_ent = pd.DataFrame(cond_ent_records)
save_analysis(df_cond_ent, "06_conditional_entropy.csv")
print(f"\nConditional entropy records: {len(df_cond_ent)}")


pythia-70m:


  draw_1: min H(Y|X) = 1.563 bits at layer 0
          info fraction = 0.327



pythia-160m:


  draw_1: min H(Y|X) = 1.415 bits at layer 6
          info fraction = 0.391



pythia-410m:


...


pythia-1b:


  draw_1: min H(Y|X) = 1.170 bits at layer 14
          info fraction = 0.496



pythia-1.4b:


  draw_1: min H(Y|X) = 1.199 bits at layer 22
          info fraction = 0.484



Conditional entropy records: 246


In [11]:
# Plot conditional entropy trajectory
for model in MODELS:
    model_data = df_cond_ent[
        (df_cond_ent["model"] == model) & (df_cond_ent["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: H(Y|X)
    axes[0].plot(
        model_data["layer"],
        model_data["h_y_given_x"],
        color="#d62728",
        marker="o",
        markersize=4,
    )
    axes[0].axhline(
        y=model_data["h_y"].iloc[0],
        color="gray",
        linestyle=":",
        alpha=0.5,
        label=f"H(Y) = {model_data['h_y'].iloc[0]:.2f}",
    )
    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("H(band | activations) (bits)")
    axes[0].set_title(f"Conditional Entropy -- {model}")
    axes[0].legend()
    axes[0].set_ylim(bottom=0)

    # Right: Information fraction
    axes[1].plot(
        model_data["layer"],
        model_data["info_fraction"],
        color="#9467bd",
        marker="o",
        markersize=4,
    )
    axes[1].axhline(y=1.0, color="gray", linestyle=":", alpha=0.3)
    axes[1].set_xlabel("Layer")
    axes[1].set_ylabel("Information Fraction (1 - H(Y|X)/H(Y))")
    axes[1].set_title(f"Fraction of Band Entropy Captured -- {model}")
    axes[1].set_ylim(-0.05, 1.05)

    fig.tight_layout()
    save_figure(fig, f"viz_06_04_conditional_entropy_{model}.png")

## 4. Delta-MI per Layer

Information gain: delta_MI(L) = MI(L) - MI(L-1). Identifies layers where the
most band-relevant processing occurs. Positive deltas indicate layers that
add information; negative deltas indicate layers that lose information.

In [12]:
delta_mi_records = []

for model in MODELS:
    for draw in DRAWS:
        # Build MI trajectory for this model/draw
        model_draw_mi = df_mi_combined[
            (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == draw)
        ].sort_values("layer")

        if len(model_draw_mi) < 2:
            continue

        # Compute delta-MI from KSG trajectory
        mi_traj = [
            {"layer": row["layer"], "mi": row["mi_ksg"]}
            for _, row in model_draw_mi.iterrows()
            if not np.isnan(row["mi_ksg"])
        ]
        if len(mi_traj) < 2:
            continue

        delta_results = compute_delta_mi(mi_traj)
        for entry in delta_results:
            delta_mi_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": entry["layer"],
                    "delta_mi": entry["delta_mi"],
                    "cumulative_mi": entry["cumulative_mi"],
                }
            )

df_delta_mi = pd.DataFrame(delta_mi_records)
save_analysis(df_delta_mi, "06_delta_mi.csv")
print(f"Delta-MI records: {len(df_delta_mi)}")

# Report peak delta-MI layers
for model in MODELS:
    model_data = df_delta_mi[
        (df_delta_mi["model"] == model) & (df_delta_mi["draw"] == "draw_1")
    ]
    if len(model_data) > 0:
        peak = model_data.loc[model_data["delta_mi"].idxmax()]
        n_layers = MODEL_INFO[model]["n_layers"]
        print(
            f"{model}: peak delta-MI = {peak['delta_mi']:.4f} bits at layer {int(peak['layer'])} "
            f"({peak['layer'] / n_layers:.1%} depth)"
        )

Delta-MI records: 246
pythia-70m: peak delta-MI = 0.7587 bits at layer 0 (0.0% depth)
pythia-160m: peak delta-MI = 0.8520 bits at layer 0 (0.0% depth)
pythia-410m: peak delta-MI = 0.8684 bits at layer 0 (0.0% depth)
pythia-1b: peak delta-MI = 1.1204 bits at layer 0 (0.0% depth)
pythia-1.4b: peak delta-MI = 0.9957 bits at layer 0 (0.0% depth)


In [13]:
# Plot delta-MI per model
for model in MODELS:
    model_data = df_delta_mi[
        (df_delta_mi["model"] == model) & (df_delta_mi["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: delta-MI bar chart
    colors = ["#2ca02c" if v >= 0 else "#d62728" for v in model_data["delta_mi"]]
    axes[0].bar(model_data["layer"], model_data["delta_mi"], color=colors, alpha=0.8)
    axes[0].axhline(y=0, color="black", linewidth=0.5)
    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("Delta MI (bits)")
    axes[0].set_title(f"Per-Layer Information Gain -- {model}")

    # Right: cumulative MI
    axes[1].plot(
        model_data["layer"],
        model_data["cumulative_mi"],
        color="#1f77b4",
        marker="o",
        markersize=4,
    )
    axes[1].axhline(
        y=H_Y, color="gray", linestyle=":", alpha=0.5, label=f"H(Y)={H_Y:.2f}"
    )
    axes[1].set_xlabel("Layer")
    axes[1].set_ylabel("Cumulative MI (bits)")
    axes[1].set_title(f"Cumulative MI -- {model}")
    axes[1].legend()
    axes[1].set_ylim(bottom=0)

    fig.tight_layout()
    save_figure(fig, f"viz_06_05_delta_mi_{model}.png")

In [14]:
# Cross-model delta-MI comparison (normalized depth)
fig, ax = plt.subplots(figsize=(14, 7))
for model in MODELS:
    model_data = df_delta_mi[
        (df_delta_mi["model"] == model) & (df_delta_mi["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    n_layers = MODEL_INFO[model]["n_layers"]
    layer_frac = model_data["layer"] / (n_layers - 1)
    ax.plot(
        layer_frac,
        model_data["delta_mi"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_xlabel("Relative Layer Position")
ax.set_ylabel("Delta MI (bits)")
ax.set_title("Per-Layer Information Gain Across Models")
ax.legend()
fig.tight_layout()
save_figure(fig, "viz_06_06_delta_mi_all_models.png")

## 5. Component-wise MI

Separate MI estimates for attention outputs vs MLP outputs at each layer.
This reveals whether band information is primarily introduced by attention
(e.g., via induction heads) or by MLPs (e.g., via frequency-dependent
transformations).

In [15]:
component_mi_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model}:")

    for draw in DRAWS:
        # Check if we have component data
        has_attn = any(
            all_attn_out.get(model, {}).get(draw, {}).get(b) is not None for b in BANDS
        )
        has_mlp = any(
            all_mlp_out.get(model, {}).get(draw, {}).get(b) is not None for b in BANDS
        )

        if not (has_attn and has_mlp):
            print(f"  {draw}: Missing attn_out or mlp_out -- skipping component MI")
            continue

        for layer in range(n_layers):
            # Attention outputs
            X_attn, labels_attn = build_layer_data(all_attn_out, model, draw, layer)
            # MLP outputs
            X_mlp, labels_mlp = build_layer_data(all_mlp_out, model, draw, layer)

            if X_attn is None or X_mlp is None:
                continue

            # Reduce dimensionality
            X_attn_r = reduce_dims(X_attn, n_components=10)
            X_mlp_r = reduce_dims(X_mlp, n_components=10)

            attn_mi = estimate_mi_ksg(X_attn_r, labels_attn, k=K_NEIGHBORS)
            mlp_mi = estimate_mi_ksg(X_mlp_r, labels_mlp, k=K_NEIGHBORS)

            component_mi_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "attn_mi": attn_mi,
                    "mlp_mi": mlp_mi,
                    "mlp_advantage": mlp_mi - attn_mi,
                }
            )

        if draw == "draw_1":
            draw_recs = [
                r
                for r in component_mi_records
                if r["model"] == model and r["draw"] == draw
            ]
            if draw_recs:
                peak_attn = max(draw_recs, key=lambda r: r["attn_mi"])
                peak_mlp = max(draw_recs, key=lambda r: r["mlp_mi"])
                print(
                    f"  {draw}: peak attn MI = {peak_attn['attn_mi']:.3f} (L{peak_attn['layer']})"
                )
                print(
                    f"          peak MLP MI  = {peak_mlp['mlp_mi']:.3f} (L{peak_mlp['layer']})"
                )

df_component_mi = pd.DataFrame(component_mi_records)
if len(df_component_mi) > 0:
    save_analysis(df_component_mi, "06_component_mi.csv")
    print(f"\nComponent MI records: {len(df_component_mi)}")
else:
    print("\nNo component MI data available (attn_out/mlp_out not extracted).")


pythia-70m:


  draw_1: peak attn MI = 1.052 (L1)
          peak MLP MI  = 0.615 (L0)



pythia-160m:


  draw_1: peak attn MI = 0.991 (L6)
          peak MLP MI  = 0.757 (L0)



pythia-410m:


...


pythia-1b:


  draw_1: peak attn MI = 1.435 (L0)
          peak MLP MI  = 1.007 (L1)



pythia-1.4b:


  draw_1: peak attn MI = 1.368 (L19)
          peak MLP MI  = 1.005 (L22)



Component MI records: 246


In [16]:
# Plot component MI per model
if len(df_component_mi) > 0:
    for model in MODELS:
        model_data = df_component_mi[
            (df_component_mi["model"] == model) & (df_component_mi["draw"] == "draw_1")
        ].sort_values("layer")
        if len(model_data) == 0:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Attn vs MLP MI trajectories
        axes[0].plot(
            model_data["layer"],
            model_data["attn_mi"],
            color="#1f77b4",
            label="Attention",
            marker="o",
            markersize=4,
        )
        axes[0].plot(
            model_data["layer"],
            model_data["mlp_mi"],
            color="#ff7f0e",
            label="MLP",
            marker="s",
            markersize=4,
        )
        axes[0].set_xlabel("Layer")
        axes[0].set_ylabel("MI (bits)")
        axes[0].set_title(f"Component MI -- {model}")
        axes[0].legend()
        axes[0].set_ylim(bottom=0)

        # Right: MLP advantage (MLP MI - Attn MI)
        colors = [
            "#ff7f0e" if v >= 0 else "#1f77b4" for v in model_data["mlp_advantage"]
        ]
        axes[1].bar(
            model_data["layer"], model_data["mlp_advantage"], color=colors, alpha=0.8
        )
        axes[1].axhline(y=0, color="black", linewidth=0.5)
        axes[1].set_xlabel("Layer")
        axes[1].set_ylabel("MLP MI - Attn MI (bits)")
        axes[1].set_title(f"MLP Information Advantage -- {model}")

        fig.tight_layout()
        save_figure(fig, f"viz_06_07_component_mi_{model}.png")
else:
    print("No component MI data available -- skipping visualization.")

## 6. Coding Efficiency

Bits of band information per dimension: efficiency = MI / d_model.
This normalizes MI by the representational capacity of each model,
allowing fair comparison across model sizes.

In [17]:
efficiency_records = []

for model in MODELS:
    d_model = MODEL_D_MODEL[model]

    for draw in DRAWS:
        model_draw_mi = df_mi_combined[
            (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == draw)
        ].sort_values("layer")

        if len(model_draw_mi) == 0:
            continue

        # Compute coding efficiency trajectory from KSG MI
        mi_traj = [
            {"layer": row["layer"], "mi": row["mi_ksg"]}
            for _, row in model_draw_mi.iterrows()
            if not np.isnan(row["mi_ksg"])
        ]

        eff_traj = compute_coding_efficiency_trajectory(mi_traj, d_model)

        for entry in eff_traj:
            efficiency_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": entry["layer"],
                    "mi": entry["mi"],
                    "efficiency": entry["efficiency"],
                    "d_model": d_model,
                    "model_capacity": MODEL_CAPACITY[model],
                }
            )

df_efficiency = pd.DataFrame(efficiency_records)
save_analysis(df_efficiency, "06_coding_efficiency.csv")
print(f"Coding efficiency records: {len(df_efficiency)}")

# Summary: peak efficiency per model
for model in MODELS:
    model_data = df_efficiency[
        (df_efficiency["model"] == model) & (df_efficiency["draw"] == "draw_1")
    ]
    if len(model_data) > 0:
        peak = model_data.loc[model_data["efficiency"].idxmax()]
        print(
            f"{model} (d={MODEL_D_MODEL[model]}): peak eff = {peak['efficiency']:.6f} bits/dim "
            f"at layer {int(peak['layer'])} (MI = {peak['mi']:.3f} bits)"
        )

Coding efficiency records: 246
pythia-70m (d=512): peak eff = 0.001482 bits/dim at layer 0 (MI = 0.759 bits)
pythia-160m (d=768): peak eff = 0.001181 bits/dim at layer 6 (MI = 0.907 bits)
pythia-410m (d=1024): peak eff = 0.000959 bits/dim at layer 3 (MI = 0.982 bits)
pythia-1b (d=2048): peak eff = 0.000562 bits/dim at layer 14 (MI = 1.152 bits)
pythia-1.4b (d=2048): peak eff = 0.000548 bits/dim at layer 22 (MI = 1.123 bits)


In [18]:
# Plot coding efficiency trajectories
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Efficiency trajectories (absolute layer)
for model in MODELS:
    model_data = df_efficiency[
        (df_efficiency["model"] == model) & (df_efficiency["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    axes[0].plot(
        model_data["layer"],
        model_data["efficiency"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Coding Efficiency (bits/dim)")
axes[0].set_title("Coding Efficiency Trajectory")
axes[0].legend()
axes[0].set_ylim(bottom=0)

# Right: Peak efficiency vs model capacity
peak_eff_data = []
for model in MODELS:
    for draw in DRAWS:
        md = df_efficiency[
            (df_efficiency["model"] == model) & (df_efficiency["draw"] == draw)
        ]
        if len(md) > 0:
            peak_eff_data.append(
                {
                    "model": model,
                    "draw": draw,
                    "model_capacity": MODEL_CAPACITY[model],
                    "d_model": MODEL_D_MODEL[model],
                    "peak_efficiency": md["efficiency"].max(),
                    "peak_mi": md.loc[md["efficiency"].idxmax(), "mi"],
                }
            )

df_peak_eff = pd.DataFrame(peak_eff_data)
for model in MODELS:
    md = df_peak_eff[df_peak_eff["model"] == model]
    if len(md) > 0:
        axes[1].scatter(
            md["model_capacity"],
            md["peak_efficiency"],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            s=60,
            zorder=5,
        )

# Trend line (mean per model)
means = (
    df_peak_eff.groupby("model")
    .agg({"model_capacity": "first", "peak_efficiency": "mean"})
    .sort_values("model_capacity")
)
axes[1].plot(means["model_capacity"], means["peak_efficiency"], "k--", alpha=0.3)

axes[1].set_xlabel("Model Capacity (M params)")
axes[1].set_ylabel("Peak Coding Efficiency (bits/dim)")
axes[1].set_title("Peak Efficiency vs Model Size")
axes[1].set_xscale("log")
axes[1].legend()

fig.tight_layout()
save_figure(fig, "viz_06_08_coding_efficiency.png")

## 7. Cross-Model Comparison

Summarize MI trajectories, delta-MI peaks, and coding efficiency scaling
across all four Pythia models.

In [19]:
# Build master info-theoretic summary
master_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_model = MODEL_D_MODEL[model]

    for draw in DRAWS:
        record = {
            "model": model,
            "draw": draw,
            "model_capacity": MODEL_CAPACITY[model],
            "n_layers": n_layers,
            "d_model": d_model,
        }

        # Peak KSG MI
        mi_data = df_mi_combined[
            (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == draw)
        ]
        if len(mi_data) > 0 and mi_data["mi_ksg"].notna().any():
            peak_idx = mi_data["mi_ksg"].idxmax()
            record["peak_mi_ksg"] = mi_data.loc[peak_idx, "mi_ksg"]
            record["peak_mi_ksg_layer"] = mi_data.loc[peak_idx, "layer"]
            record["peak_mi_ksg_layer_frac"] = mi_data.loc[peak_idx, "layer"] / (
                n_layers - 1
            )

        # Peak probe MI
        if len(mi_data) > 0 and mi_data["mi_probe"].notna().any():
            peak_idx = mi_data["mi_probe"].idxmax()
            record["peak_mi_probe"] = mi_data.loc[peak_idx, "mi_probe"]
            record["peak_mi_probe_layer"] = mi_data.loc[peak_idx, "layer"]

        # Peak delta-MI
        delta_data = df_delta_mi[
            (df_delta_mi["model"] == model) & (df_delta_mi["draw"] == draw)
        ]
        if len(delta_data) > 0:
            peak_idx = delta_data["delta_mi"].idxmax()
            record["peak_delta_mi"] = delta_data.loc[peak_idx, "delta_mi"]
            record["peak_delta_mi_layer"] = delta_data.loc[peak_idx, "layer"]
            record["peak_delta_mi_layer_frac"] = delta_data.loc[peak_idx, "layer"] / (
                n_layers - 1
            )

        # Peak coding efficiency
        eff_data = df_efficiency[
            (df_efficiency["model"] == model) & (df_efficiency["draw"] == draw)
        ]
        if len(eff_data) > 0:
            record["peak_efficiency"] = eff_data["efficiency"].max()

        # Min conditional entropy
        cond_data = df_cond_ent[
            (df_cond_ent["model"] == model) & (df_cond_ent["draw"] == draw)
        ]
        if len(cond_data) > 0:
            record["min_h_y_given_x"] = cond_data["h_y_given_x"].min()
            record["max_info_fraction"] = cond_data["info_fraction"].max()

        master_records.append(record)

df_master_info = pd.DataFrame(master_records)
save_analysis(df_master_info, "06_master_info_theoretic.csv")

# Display summary table
summary_cols = [
    "model",
    "peak_mi_ksg",
    "peak_mi_ksg_layer_frac",
    "peak_delta_mi",
    "peak_delta_mi_layer_frac",
    "peak_efficiency",
    "max_info_fraction",
]
existing_cols = [c for c in summary_cols if c in df_master_info.columns]
print("Cross-model info-theoretic summary (mean over draws):")
print(df_master_info.groupby("model")[existing_cols[1:]].mean().round(4))

Cross-model info-theoretic summary (mean over draws):
             peak_mi_ksg  peak_mi_ksg_layer_frac  peak_delta_mi  \
model                                                             
pythia-1.4b       1.1321                  0.3623         0.9903   
pythia-160m       0.9117                  0.5455         0.8647   
pythia-1b         1.1577                  0.6444         1.1363   
pythia-410m       0.9879                  0.1014         0.8745   
pythia-70m        0.7251                  0.0000         0.7251   

             peak_delta_mi_layer_frac  peak_efficiency  max_info_fraction  
model                                                                      
pythia-1.4b                       0.0           0.0006             0.4876  
pythia-160m                       0.0           0.0012             0.3927  
pythia-1b                         0.0           0.0006             0.4986  
pythia-410m                       0.0           0.0010             0.4254  
pythia-70m          

In [20]:
# Scaling panel: peak MI, peak delta-MI, peak efficiency vs model size
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scaling_metrics = [
    ("peak_mi_ksg", "Peak MI (KSG, bits)"),
    ("peak_delta_mi", "Peak Delta-MI (bits)"),
    ("peak_efficiency", "Peak Coding Efficiency (bits/dim)"),
]

for ax, (metric, title) in zip(axes, scaling_metrics):
    if metric not in df_master_info.columns:
        ax.set_title(f"{title} (N/A)")
        continue
    for model in MODELS:
        md = df_master_info[df_master_info["model"] == model]
        if len(md) == 0 or metric not in md.columns:
            continue
        ax.scatter(
            md["model_capacity"],
            md[metric],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            s=60,
            zorder=5,
        )
    # Mean trend line
    means = (
        df_master_info.groupby("model")
        .agg({"model_capacity": "first", metric: "mean"})
        .dropna()
        .sort_values("model_capacity")
    )
    if len(means) > 1:
        ax.plot(means["model_capacity"], means[metric], "k--", alpha=0.3)
    ax.set_xlabel("Model Capacity (M params)")
    ax.set_xscale("log")
    ax.set_title(title)

axes[0].legend(loc="best", fontsize=8)
fig.tight_layout()
save_figure(fig, "viz_06_09_scaling_panel.png")

In [21]:
# MI heatmap: model x layer (KSG MI, draw_1)
# Build matrix for heatmap
mi_matrix_data = {}
max_layers = max(MODEL_INFO[m]["n_layers"] for m in MODELS)

for model in MODELS:
    md = df_mi_combined[
        (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == "draw_1")
    ].sort_values("layer")
    mi_vals = np.full(max_layers, np.nan)
    for _, row in md.iterrows():
        layer_idx = int(row["layer"])
        if layer_idx < max_layers:
            mi_vals[layer_idx] = row["mi_ksg"]
    mi_matrix_data[model] = mi_vals

df_mi_matrix = pd.DataFrame(mi_matrix_data, index=range(max_layers)).T
df_mi_matrix = df_mi_matrix.reindex(MODELS)
df_mi_matrix.columns = [f"L{i}" for i in range(max_layers)]

# Drop columns that are all NaN
df_mi_matrix_clean = df_mi_matrix.dropna(axis=1, how="all")

fig, ax = plt.subplots(figsize=(max(14, max_layers * 0.6), 4))
sns.heatmap(
    df_mi_matrix_clean,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    square=True,
    linewidths=0,
    linecolor="none",
    ax=ax,
    vmin=0,
)
ax.set_xlabel("Layer")
ax.set_ylabel("Model")
ax.set_title("MI(activations; band) Heatmap (KSG, draw_1)")
fig.tight_layout()
save_figure(fig, "viz_06_10_mi_heatmap.png")

## 8. Integration with Geometric Analyses

Load results from NB02 (probe accuracy trajectory, separation ratio trajectory)
and correlate with MI-based measures. This tests whether information-theoretic
and geometric perspectives converge on the same conclusions.

In [22]:
# Load NB02 results
try:
    df_probe_traj = pd.read_csv(ANALYSIS_DIR / "02_probe_trajectory.csv")
    print(f"Loaded probe trajectory: {len(df_probe_traj)} rows")
    print(f"  Columns: {list(df_probe_traj.columns)}")
except FileNotFoundError:
    df_probe_traj = pd.DataFrame()
    print("NB02 probe trajectory not found -- skipping integration.")

try:
    df_sep_traj = pd.read_csv(ANALYSIS_DIR / "02_separation_trajectory.csv")
    print(f"Loaded separation trajectory: {len(df_sep_traj)} rows")
    print(f"  Columns: {list(df_sep_traj.columns)}")
except FileNotFoundError:
    df_sep_traj = pd.DataFrame()
    print("NB02 separation trajectory not found -- skipping integration.")

NB02 probe trajectory not found -- skipping integration.
NB02 separation trajectory not found -- skipping integration.


In [23]:
# Correlation: MI(KSG) vs probe accuracy trajectory
integration_records = []

if len(df_probe_traj) > 0:
    for model in MODELS:
        for draw in DRAWS:
            mi_data = df_mi_combined[
                (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == draw)
            ][["layer", "mi_ksg"]].dropna()

            probe_data = df_probe_traj[
                (df_probe_traj["model"] == model) & (df_probe_traj["draw"] == draw)
            ][["layer", "accuracy"]].dropna()

            if len(mi_data) < 3 or len(probe_data) < 3:
                continue

            # Merge on layer
            merged = mi_data.merge(probe_data, on="layer")
            if len(merged) < 3:
                continue

            r, p = sp_stats.pearsonr(merged["mi_ksg"], merged["accuracy"])
            rho, p_rho = sp_stats.spearmanr(merged["mi_ksg"], merged["accuracy"])

            integration_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "comparison": "MI_ksg_vs_probe_accuracy",
                    "pearson_r": r,
                    "pearson_p": p,
                    "spearman_rho": rho,
                    "spearman_p": p_rho,
                    "n_layers": len(merged),
                }
            )

    print("MI(KSG) vs Probe Accuracy:")
    for rec in integration_records:
        if rec["comparison"] == "MI_ksg_vs_probe_accuracy":
            sig = "*" if rec["pearson_p"] < 0.05 else ""
            print(
                f"  {rec['model']}/{rec['draw']}: r={rec['pearson_r']:.3f}{sig}, "
                f"rho={rec['spearman_rho']:.3f}"
            )
else:
    print("Probe trajectory not available -- skipping MI vs probe correlation.")

Probe trajectory not available -- skipping MI vs probe correlation.


In [24]:
# Correlation: MI(KSG) vs separation ratio trajectory
if len(df_sep_traj) > 0:
    for model in MODELS:
        for draw in DRAWS:
            mi_data = df_mi_combined[
                (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == draw)
            ][["layer", "mi_ksg"]].dropna()

            sep_data = df_sep_traj[
                (df_sep_traj["model"] == model) & (df_sep_traj["draw"] == draw)
            ][["layer", "separation_ratio"]].dropna()

            if len(mi_data) < 3 or len(sep_data) < 3:
                continue

            merged = mi_data.merge(sep_data, on="layer")
            if len(merged) < 3:
                continue

            r, p = sp_stats.pearsonr(merged["mi_ksg"], merged["separation_ratio"])
            rho, p_rho = sp_stats.spearmanr(
                merged["mi_ksg"], merged["separation_ratio"]
            )

            integration_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "comparison": "MI_ksg_vs_separation_ratio",
                    "pearson_r": r,
                    "pearson_p": p,
                    "spearman_rho": rho,
                    "spearman_p": p_rho,
                    "n_layers": len(merged),
                }
            )

    print("\nMI(KSG) vs Separation Ratio:")
    for rec in integration_records:
        if rec["comparison"] == "MI_ksg_vs_separation_ratio":
            sig = "*" if rec["pearson_p"] < 0.05 else ""
            print(
                f"  {rec['model']}/{rec['draw']}: r={rec['pearson_r']:.3f}{sig}, "
                f"rho={rec['spearman_rho']:.3f}"
            )
else:
    print(
        "Separation trajectory not available -- skipping MI vs separation correlation."
    )

Separation trajectory not available -- skipping MI vs separation correlation.


In [25]:
# Save integration results
if integration_records:
    df_integration = pd.DataFrame(integration_records)
    save_analysis(df_integration, "06_geometric_integration.csv")
    print(f"Integration records: {len(df_integration)}")
    print("\nSummary by comparison type:")
    print(
        df_integration.groupby("comparison")[["pearson_r", "spearman_rho"]]
        .mean()
        .round(3)
    )

In [26]:
# Visualization: MI vs probe accuracy scatter (per model)
if len(df_probe_traj) > 0:
    fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5), sharey=True)
    if len(MODELS) == 1:
        axes = [axes]

    for ax, model in zip(axes, MODELS):
        mi_data = df_mi_combined[
            (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == "draw_1")
        ][["layer", "mi_ksg"]].dropna()

        probe_data = df_probe_traj[
            (df_probe_traj["model"] == model) & (df_probe_traj["draw"] == "draw_1")
        ][["layer", "accuracy"]].dropna()

        merged = mi_data.merge(probe_data, on="layer")
        if len(merged) < 2:
            ax.set_title(f"{model} (insufficient data)")
            continue

        scatter = ax.scatter(
            merged["mi_ksg"],
            merged["accuracy"],
            c=merged["layer"],
            cmap="viridis",
            s=50,
            edgecolors="black",
            linewidth=0.5,
        )
        plt.colorbar(scatter, ax=ax, label="Layer")

        # Add correlation
        if len(merged) > 2:
            r, _ = sp_stats.pearsonr(merged["mi_ksg"], merged["accuracy"])
            ax.set_title(f"{model} (r={r:.3f})")
        else:
            ax.set_title(model)

        ax.set_xlabel("MI (KSG, bits)")
        if ax == axes[0]:
            ax.set_ylabel("Probe Accuracy")

    fig.suptitle("MI(KSG) vs Probe Accuracy (colored by layer)", y=1.02)
    fig.tight_layout()
    save_figure(fig, "viz_06_11_mi_vs_probe_scatter.png")
else:
    print("Probe trajectory not available -- skipping scatter plot.")

Probe trajectory not available -- skipping scatter plot.


In [27]:
# Overlay plot: MI and separation ratio on dual y-axes (draw_1)
if len(df_sep_traj) > 0:
    for model in MODELS:
        mi_data = df_mi_combined[
            (df_mi_combined["model"] == model) & (df_mi_combined["draw"] == "draw_1")
        ].sort_values("layer")
        sep_data = df_sep_traj[
            (df_sep_traj["model"] == model) & (df_sep_traj["draw"] == "draw_1")
        ].sort_values("layer")

        if len(mi_data) == 0 or len(sep_data) == 0:
            continue

        fig, ax1 = plt.subplots(figsize=(12, 6))
        color1 = "#1f77b4"
        ax1.plot(
            mi_data["layer"],
            mi_data["mi_ksg"],
            color=color1,
            marker="o",
            markersize=4,
            label="MI (KSG)",
        )
        ax1.set_xlabel("Layer")
        ax1.set_ylabel("MI (bits)", color=color1)
        ax1.tick_params(axis="y", labelcolor=color1)
        ax1.set_ylim(bottom=0)

        ax2 = ax1.twinx()
        color2 = "#d62728"
        ax2.plot(
            sep_data["layer"],
            sep_data["separation_ratio"],
            color=color2,
            marker="s",
            markersize=4,
            label="Separation Ratio",
        )
        ax2.set_ylabel("Separation Ratio", color=color2)
        ax2.tick_params(axis="y", labelcolor=color2)
        ax2.set_ylim(bottom=0)

        # Combined legend
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

        fig.suptitle(f"MI vs Separation Ratio -- {model}")
        fig.tight_layout()
        save_figure(fig, f"viz_06_12_mi_vs_separation_{model}.png")
else:
    print("Separation trajectory not available -- skipping overlay plots.")

Separation trajectory not available -- skipping overlay plots.


## 9. Summary

In [28]:
# Draw stability analysis
stability_records = []
for model in MODELS:
    model_data = df_master_info[df_master_info["model"] == model]
    if len(model_data) < 2:
        continue

    stability_metrics = [
        "peak_mi_ksg",
        "peak_mi_ksg_layer_frac",
        "peak_delta_mi",
        "peak_delta_mi_layer_frac",
        "peak_efficiency",
        "max_info_fraction",
    ]

    for metric in stability_metrics:
        if metric not in model_data.columns:
            continue
        vals = model_data[metric].dropna()
        if len(vals) < 2:
            continue
        stability_records.append(
            {
                "model": model,
                "metric": metric,
                "mean": float(vals.mean()),
                "std": float(vals.std()),
                "cv": float(vals.std() / vals.mean()) if vals.mean() != 0 else np.nan,
            }
        )

df_stability = pd.DataFrame(stability_records)
if len(df_stability) > 0:
    save_analysis(df_stability, "06_draw_stability.csv")
    print("Draw stability (CV = coefficient of variation):")
    pivot = df_stability.pivot(index="model", columns="metric", values="cv")
    print(pivot.round(4))
else:
    print("Insufficient data for stability analysis.")

Draw stability (CV = coefficient of variation):
metric       max_info_fraction  peak_delta_mi  peak_delta_mi_layer_frac  \
model                                                                     
pythia-1.4b             0.0225         0.0104                       NaN   
pythia-160m             0.0046         0.0194                       NaN   
pythia-1b               0.0082         0.0133                       NaN   
pythia-410m             0.0237         0.0163                       NaN   
pythia-70m              0.0490         0.0490                       NaN   

metric       peak_efficiency  peak_mi_ksg  peak_mi_ksg_layer_frac  
model                                                              
pythia-1.4b           0.0225       0.0225                  1.4215  
pythia-160m           0.0046       0.0046                  0.0000  
pythia-1b             0.0082       0.0082                  0.7764  
pythia-410m           0.0237       0.0237                  0.4949  
pythia-70m        

In [29]:
print("\n" + "=" * 70)
print("NOTEBOOK 06 COMPLETE: Information-Theoretic Analysis")
print("=" * 70)

print(f"\nOutput CSVs in: {ANALYSIS_DIR}")
for f in sorted(ANALYSIS_DIR.glob("06_*")):
    print(f"  {f.name}")

print(f"\nFigures in: {VIZ_DIR}")
for f in sorted(VIZ_DIR.glob("viz_06_*")):
    print(f"  {f.name}")

print("\n--- Key Findings ---")
for model in MODELS:
    md = df_master_info[
        (df_master_info["model"] == model) & (df_master_info["draw"] == "draw_1")
    ]
    if len(md) == 0:
        continue
    row = md.iloc[0]
    lines = [f"\n{model} (d_model={MODEL_D_MODEL[model]}):"]
    if "peak_mi_ksg" in row and not np.isnan(row.get("peak_mi_ksg", np.nan)):
        lines.append(
            f"  Peak MI (KSG)       = {row['peak_mi_ksg']:.3f} bits "
            f"at layer {int(row['peak_mi_ksg_layer'])} "
            f"({row['peak_mi_ksg_layer_frac']:.0%} depth)"
        )
    if "peak_delta_mi" in row and not np.isnan(row.get("peak_delta_mi", np.nan)):
        lines.append(
            f"  Peak delta-MI       = {row['peak_delta_mi']:.4f} bits "
            f"at layer {int(row['peak_delta_mi_layer'])}"
        )
    if "peak_efficiency" in row and not np.isnan(row.get("peak_efficiency", np.nan)):
        lines.append(f"  Peak efficiency     = {row['peak_efficiency']:.6f} bits/dim")
    if "max_info_fraction" in row and not np.isnan(
        row.get("max_info_fraction", np.nan)
    ):
        lines.append(f"  Max info fraction   = {row['max_info_fraction']:.3f}")
    print("\n".join(lines))

# Report integration correlations
if integration_records:
    print("\n--- Geometric Integration ---")
    df_int = pd.DataFrame(integration_records)
    for comparison in df_int["comparison"].unique():
        cd = df_int[df_int["comparison"] == comparison]
        mean_r = cd["pearson_r"].mean()
        mean_rho = cd["spearman_rho"].mean()
        print(f"  {comparison}: mean r={mean_r:.3f}, mean rho={mean_rho:.3f}")


NOTEBOOK 06 COMPLETE: Information-Theoretic Analysis

Output CSVs in: LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/base/analysis
  06_coding_efficiency.csv
  06_component_mi.csv
  06_conditional_entropy.csv
  06_delta_mi.csv
  06_draw_stability.csv
  06_geometric_integration.csv
  06_master_info_theoretic.csv
  06_mi_combined_trajectory.csv
  06_mi_ksg_trajectory.csv
  06_mi_probe_trajectory.csv

Figures in: LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/base/viz
  viz_06_01_mi_trajectory_pythia-1.4b.png
  viz_06_01_mi_trajectory_pythia-160m.png
  viz_06_01_mi_trajectory_pythia-1b.png
  viz_06_01_mi_trajectory_pythia-410m.png
  viz_06_01_mi_trajectory_pythia-70m.png
  viz_06_02_mi_trajectory_all_models.png
  viz_06_03_mi_ksg_vs_probe_scatter.png
  viz_06_04_conditional_entropy_pythia-1.4b.png
  viz_06_04_conditional_entropy_pythia-160m.png
  viz_06_04_conditional_entropy_pythia-1b.png
  viz_06_04_conditional_entropy_pythia-410m.png
  viz